# Free Colab: preceding-source-context experiment

Select a free GPU runtime and run cells in order. Upload `reranker_parent_colab_inputs.zip`.
The model downloads inside Colab, never to your laptop. No credentials are needed.
Treatment: source/section headers plus one immediately preceding passage for short candidates
when document/language/location/access roles match and section titles do not conflict.
Control is metadata-only; treatment adds preceding source text. This isolates one added factor.
JSON uses legacy 'plain' keys for the control; config.control_mode explicitly records 'metadata'.
Same pinned model, E5 top30, 80 cases and evidence v3; both arms run on this GPU.
Only original selected candidate text earns evidence coverage; neighboring text is ranking-only.
There is no total context-token cap. Inputs exceeding the per-pair model limit fail preflight.
This post-hoc diagnostic is not held-out evaluation.


In [ ]:
%pip -q install sentence-transformers==6.0.1 transformers==5.17.0 huggingface-hub==1.31.0 numpy==2.4.6 python-dotenv
# If Colab asks to restart the session, restart and continue from the next cell.


In [ ]:
import os
os.environ.update(OMP_NUM_THREADS='4', MKL_NUM_THREADS='4', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
import torch
assert torch.cuda.is_available(), 'No GPU assigned. Select a free GPU runtime; this notebook will not silently run on CPU.'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # Select reranker_parent_colab_inputs.zip from your laptop.
assert 'reranker_parent_colab_inputs.zip' in uploaded, 'Upload the input bundle, not the notebook.'
work = Path('/content/reranker_parent_experiment')
work.mkdir(exist_ok=True)
with zipfile.ZipFile('reranker_parent_colab_inputs.zip') as bundle:
    assert all((work / name).resolve().is_relative_to(work.resolve()) for name in bundle.namelist())
    bundle.extractall(work)
del uploaded
os.chdir(work)
print('Inputs extracted. No credentials were included.')


In [ ]:
import subprocess, sys
output = Path('/content/reranker_parent_results')
base = [sys.executable, '-X', 'utf8', 'scripts/retrieval/evaluate_reranker_source_context.py',
        '--input-root', str(work), '--output-dir', str(output), '--context-mode', 'parent']

# First downloads the pinned model to Colab, then checks input lengths and runs both arms.
# A completed output folder is protected. To repeat, choose a NEW output folder.
subprocess.run(base + ['--device', 'cuda', '--allow-download'], check=True)


In [ ]:
# Verify all saved rankings and evidence scores without loading the model again.
subprocess.run(base + ['--verify-only'], check=True)

import json
r = json.loads((output / 'results.json').read_text())
print('Direction | Metadata-only complete intents | Parent-context complete intents')
for direction in ('vi->vi', 'en->en', 'vi->en', 'en->vi'):
    a, b = [r['summary'][arm][direction] for arm in ('plain', 'source_context')]
    print(direction, f"{a['full_evidence_cases_at_10']}/{a['full_intent_eligible_cases']}",
          f"{b['full_evidence_cases_at_10']}/{b['full_intent_eligible_cases']}")


In [ ]:
# Download result artifacts only: no model weights, token, or source-data cache.
import shutil
assert output.exists(), 'Run the experiment cell first.'
archive = shutil.make_archive('/content/reranker_parent_context_results', 'zip', root_dir=output)
files.download(archive)
# Return this ZIP to the project/Codex for Q13/Q15 and regression analysis.
# If inference failed, you can still use this cell to save partial checkpoints;
# config.json will identify incomplete runs. They are NOT completed results.
